# Very first script to run - OBS VEGA data only
## simple script to extract and massage only the necessary data
### Water, soil, sediment only, not Air
Alexander Minidis

update 2025-12-01 VEGA data only

Note: Smiles fixing via Opsin (Pubchem) or manual intervention not necessary in this case since all structures were clean.


In [1]:
import pandas as pd
from pandas import read_csv
from pathlib import Path
import requests
from tqdm.notebook import tqdm

pd.set_option("display.max_columns", None)
pd.set_option("display.width", 200)

In [2]:
# set directories and filenames, load data
working_dir = Path.cwd().parent
raw_data_dir = working_dir / "raw_data" / "VEGA_paper_data"
output_dir = working_dir / "processed_data"

soil_file = raw_data_dir / "dataset_PERS_SOIL_QUANT.txt"
water_file = raw_data_dir / "dataset_PERS_WATER_QUANT.txt"
sediment_file = raw_data_dir / "dataset_PERS_SED_QUANT.txt"


soil_data_all = read_csv(soil_file, index_col=0, header=0, sep="\t")
water_data_all = read_csv(water_file, index_col=0, header=0, sep="\t")
sediment_data_all = read_csv(sediment_file, index_col=0, header=0, sep="\t")

soil_data_all = soil_data_all.loc[:, ~soil_data_all.columns.isin(["Status", "Predicted value  [log(days)]"])]
water_data_all = water_data_all.loc[:, ~water_data_all.columns.isin(["Status", "Predicted value "])]
sediment_data_all = sediment_data_all.loc[:, ~sediment_data_all.columns.isin(["Status", "Predicted value "])]

In [3]:
# adapt column names to standard format and convert log days to days for half-life
soil_data_all = soil_data_all.rename(
    columns={"Compound name": "Compound", "Experimental value  [log(days)]": "T_half_soil_days", "SMILES": "Canonical_smiles"}
)
soil_data_all["T_half_soil_days"] = 10 ** soil_data_all["T_half_soil_days"]

water_data_all = water_data_all.rename(
    columns={"Compound name": "Compound", "Experimental value ": "T_half_water_days", "SMILES": "Canonical_smiles"}
)
water_data_all["T_half_water_days"] = 10 ** water_data_all["T_half_water_days"]

sediment_data_all = sediment_data_all.rename(
    columns={"Compound name": "Compound", "Experimental value ": "T_half_sediment_days", "SMILES": "Canonical_smiles"}
)
sediment_data_all["T_half_sediment_days"] = 10 ** sediment_data_all["T_half_sediment_days"]

In [4]:
print(soil_data_all.head())

          CAS                                   Canonical_smiles  T_half_soil_days
Id                                                                                
1     74-97-5                                            C(Cl)Br         70.833337
2   2385-85-5  C13(C4(C2(C5(C(C1(C2(Cl)Cl)Cl)(C3(C(C45Cl)(Cl)...       2291.666942
3   7085-19-0                           O=C(O)C(Oc1ccc(cc1C)Cl)C          7.083333
4     95-93-2                                 c1c(c(cc(c1C)C)C)C         70.833337
5     56-38-2               O=[N+]([O-])c1ccc(OP(OCC)(OCC)=S)cc1         22.916664


In [5]:
soil_data = soil_data_all.copy()
water_data = water_data_all.copy()
sediment_data = sediment_data_all.copy()

In [6]:
dataframe_list = [soil_data, water_data, sediment_data]

In [7]:
print(
    f"Soil data shape: {soil_data_all.shape}"
    f"\nWater data shape: {water_data_all.shape}"
    f"\nSediment data shape: {sediment_data_all.shape}"
)
soil_data.dropna(subset=["Canonical_smiles"], inplace=True)
water_data.dropna(subset=["Canonical_smiles"], inplace=True)
sediment_data.dropna(subset=["Canonical_smiles"], inplace=True)
print(
    f"After dropping entries with missing SMILES:"
    f"\nSoil data shape: {soil_data.shape}"
    f"\nWater data shape: {water_data.shape}"
    f"\nSediment data shape: {sediment_data.shape}"
)

Soil data shape: (226, 3)
Water data shape: (223, 3)
Sediment data shape: (221, 3)
After dropping entries with missing SMILES:
Soil data shape: (226, 3)
Water data shape: (223, 3)
Sediment data shape: (221, 3)


No missing entries, further analysis not required

In [8]:
# Final save of processed data
soil_data.to_csv(output_dir / "vega_soil_t_half_data.csv", index=False, sep="\t")
water_data.to_csv(output_dir / "vega_water_t_half_data.csv", index=False, sep="\t")
sediment_data.to_csv(output_dir / "vega_sediment_t_half_data.csv", index=False, sep="\t")